# 13주차 Type III 실습
## 로지스틱 회귀(Logistic Regression) 실습 노트

### 실습 목표
- 이진 종속변수 문제에서 로지스틱 회귀를 사용할 수 있다.
- 회귀계수, p-value, 오즈비(odds ratio)를 해석할 수 있다.
- log-likelihood, pseudo R², residual deviance, accuracy, error rate를 확인할 수 있다.
- `churn_logit1.csv`, `loan_default_logit2.csv`를 분석하여 보고서 문장을 작성할 수 있다.

### 사용 파일
- `churn_logit1.csv`
- `loan_default_logit2.csv`

## 1. 왜 로지스틱 회귀를 사용하는가?

다중회귀는 종속변수가 연속형일 때 사용한다.  
하지만 이번 실습의 종속변수는 `churn(이탈 여부)`, `default(연체 여부)`처럼 0/1 값이다.  
이 경우에는 사건 발생 확률을 예측하는 **로지스틱 회귀**를 사용해야 한다.

### 이번 시간 핵심 해석 포인트
1. **회귀계수(coef)**: 사건 발생 방향이 증가인지 감소인지
2. **p-value**: 해당 변수가 유의한지
3. **오즈비(exp(coef))**: 변수 1단위 증가 시 사건 odds가 몇 배가 되는지
4. **log-likelihood / residual deviance**: 모형 적합도 확인
5. **accuracy / error rate**: 분류 성능 확인

In [ ]:
# Google Drive 마운트

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 공통 라이브러리 import

from pathlib import Path
import numpy as np
import pandas as pd

import statsmodels.formula.api as smf
from sklearn.metrics import accuracy_score, confusion_matrix

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

In [ ]:
# 데이터 경로 설정 및 파일 존재 여부 확인

DATA_DIR = Path('/content/drive/MyDrive/type3_week13')  # 필요시 수정

required_files = [
    'churn_logit1.csv',
    'loan_default_logit2.csv'
]

print("=== 파일 확인 ===")
for f in required_files:
    file_path = DATA_DIR / f
    print(f"{f}: {'존재함' if file_path.exists() else '없음'}")

## 2. 공통 해석 함수 만들기

아래 함수는 로지스틱 회귀 결과를 조금 더 보기 쉽게 정리해 준다.

### 출력 내용
- 회귀계수
- p-value
- 오즈비
- 신뢰구간
- 예측확률
- accuracy / error rate
- confusion matrix
- log-likelihood
- residual deviance

In [ ]:
def logistic_analysis_report(model, df, y_col, threshold=0.5):
    result_table = pd.DataFrame({
        'coef': model.params,
        'p_value': model.pvalues,
        'odds_ratio': np.exp(model.params)
    })

    conf = model.conf_int()
    conf.columns = ['2.5%', '97.5%']
    conf['OR_2.5%'] = np.exp(conf['2.5%'])
    conf['OR_97.5%'] = np.exp(conf['97.5%'])

    df_result = df.copy()
    df_result['pred_prob'] = model.predict(df_result)
    df_result['pred_class'] = (df_result['pred_prob'] >= threshold).astype(int)

    acc = accuracy_score(df_result[y_col], df_result['pred_class'])
    err = 1 - acc
    cm = confusion_matrix(df_result[y_col], df_result['pred_class'])

    llf = model.llf
    llnull = model.llnull
    pseudo_r2 = model.prsquared
    residual_deviance = -2 * llf

    print("=== 회귀 요약표 ===")
    print(model.summary())

    print("
=== 계수 / p-value / 오즈비 ===")
    print(result_table)

    print("
=== 95% 신뢰구간 및 오즈비 신뢰구간 ===")
    print(conf)

    print("
=== 모형 적합도 ===")
    print(f"log-likelihood      : {llf:.4f}")
    print(f"null log-likelihood : {llnull:.4f}")
    print(f"pseudo R^2          : {pseudo_r2:.4f}")
    print(f"residual deviance   : {residual_deviance:.4f}")

    print("
=== 분류 성능 ===")
    print(f"accuracy   : {acc:.4f}")
    print(f"error rate : {err:.4f}")

    print("
=== confusion matrix ===")
    print(cm)

    print("
=== 예측확률 상위 10개 ===")
    print(df_result[[y_col, 'pred_prob', 'pred_class']].head(10))

    return result_table, conf, df_result, acc, err, residual_deviance

# 실습 1. churn_logit1.csv 분석

## 문제 상황
구독형 서비스 기업이 고객 이탈 여부(`churn`)를 예측하고자 한다.  
설명변수는 다음과 같다.

- `age`: 고객 나이
- `usage_hour`: 월 이용시간
- `complaint_cnt`: 최근 불만 건수

## 분석 목표
- 어떤 변수가 고객 이탈에 영향을 주는지 확인한다.
- 각 변수의 방향성과 유의성을 해석한다.
- 오즈비를 이용해 실무적으로 설명한다.

## 가설적 해석 방향
- `complaint_cnt`가 증가하면 이탈 가능성이 커질 수 있다.
- `usage_hour`가 많을수록 서비스 충성도가 높아 이탈 가능성이 낮아질 수 있다.

In [ ]:
# churn_logit1.csv 불러오기 및 데이터 확인

churn_df = pd.read_csv(DATA_DIR / 'churn_logit1.csv')

print("=== 데이터 상위 5행 ===")
display(churn_df.head())

print("
=== 데이터 구조 ===")
print(churn_df.info())

print("
=== 기술통계 ===")
display(churn_df.describe())

print("
=== 종속변수 분포 ===")
display(churn_df['churn'].value_counts().sort_index())

In [ ]:
# 로지스틱 회귀 적합: churn ~ age + usage_hour + complaint_cnt

churn_model = smf.logit(
    formula='churn ~ age + usage_hour + complaint_cnt',
    data=churn_df
).fit()

churn_result_table, churn_conf, churn_pred_df, churn_acc, churn_err, churn_dev = logistic_analysis_report(
    churn_model, churn_df, 'churn'
)

In [ ]:
# 변수별 해석 문장 자동 출력

print("=== 변수별 해석 ===")
for var in ['age', 'usage_hour', 'complaint_cnt']:
    coef = churn_model.params[var]
    pval = churn_model.pvalues[var]
    odds = np.exp(coef)

    direction = "증가" if coef > 0 else "감소"

    print(f"
[{var}]")
    print(f"- 계수: {coef:.4f}")
    print(f"- p-value: {pval:.6f}")
    print(f"- 오즈비: {odds:.4f}")

    if pval < 0.05:
        print(f"→ {var}는 유의한 변수이다.")
    else:
        print(f"→ {var}는 유의하다고 보기 어렵다.")

    print(f"→ {var}가 1단위 증가할 때 이탈 odds는 약 {odds:.4f}배가 되며, 방향은 {direction}이다.")

## 실습 1 결과 해석 가이드

샘플 CSV 기준으로는 다음과 같은 해석이 가능하다.

- `age`는 p-value가 0.05보다 커서 유의하지 않을 수 있다.
- `usage_hour`는 음(-)의 계수로 나타나면, 이용시간이 많을수록 이탈 가능성이 낮아진다고 해석한다.
- `complaint_cnt`는 양(+)의 계수이면서 유의하면, 불만 건수가 많을수록 이탈 가능성이 높아진다고 해석한다.
- 특히 `exp(coef)` 값이 1보다 크면 odds 증가, 1보다 작으면 odds 감소로 해석한다.

### 보고서 문장 예시
“로지스틱 회귀분석 결과, `usage_hour`와 `complaint_cnt`는 고객 이탈 여부에 유의한 영향을 미쳤고, `age`는 유의하지 않았다. 특히 `complaint_cnt`의 오즈비가 1보다 크게 나타나 불만 건수가 증가할수록 고객 이탈 odds가 증가한다고 해석할 수 있다. 반면 `usage_hour`는 음의 계수로 나타나 서비스 이용시간이 많을수록 이탈 가능성이 낮아지는 경향을 보였다.”

## 학생 확인 질문 1

1. `complaint_cnt`의 계수가 양수이면 무엇을 의미하는가?  
2. `usage_hour`의 오즈비가 1보다 작다면 어떻게 해석하는가?  
3. `age`의 p-value가 0.05보다 크면 어떤 결론을 내려야 하는가?  
4. accuracy가 높다는 것은 무엇을 의미하는가?  

# 실습 2. loan_default_logit2.csv 분석

## 문제 상황
은행이 고객의 대출 연체 여부를 예측하고자 한다.  
설명변수는 다음과 같다.

- `income`: 소득
- `debt_ratio`: 부채비율
- `late_cnt`: 과거 연체 횟수

종속변수는 `default`이며, 1이면 연체 발생, 0이면 정상 상환으로 본다.

## 분석 목표
- 어떤 변수가 연체 여부에 영향을 주는지 확인한다.
- 각 변수의 오즈비를 통해 실무적으로 해석한다.
- log-likelihood, residual deviance, accuracy, error rate를 확인한다.

In [ ]:
# loan_default_logit2.csv 불러오기 및 데이터 확인

loan_df = pd.read_csv(DATA_DIR / 'loan_default_logit2.csv')

print("=== 원본 데이터 상위 5행 ===")
display(loan_df.head())

print("
=== 데이터 구조 ===")
print(loan_df.info())

print("
=== 기술통계 ===")
display(loan_df.describe())

print("
=== 종속변수 분포 ===")
display(loan_df['default'].value_counts().sort_index())

In [ ]:
# 'default'는 Python 예약어와 헷갈릴 수 있으므로 안전하게 이름 변경
loan_df = loan_df.rename(columns={'default': 'default_flag'})

print("변경된 컬럼명:")
print(loan_df.columns.tolist())

In [ ]:
# 로지스틱 회귀 적합: default_flag ~ income + debt_ratio + late_cnt

loan_model = smf.logit(
    formula='default_flag ~ income + debt_ratio + late_cnt',
    data=loan_df
).fit()

loan_result_table, loan_conf, loan_pred_df, loan_acc, loan_err, loan_dev = logistic_analysis_report(
    loan_model, loan_df, 'default_flag'
)

In [ ]:
# 변수별 해석 문장 자동 출력

print("=== 변수별 해석 ===")
for var in ['income', 'debt_ratio', 'late_cnt']:
    coef = loan_model.params[var]
    pval = loan_model.pvalues[var]
    odds = np.exp(coef)

    direction = "증가" if coef > 0 else "감소"

    print(f"
[{var}]")
    print(f"- 계수: {coef:.6f}")
    print(f"- p-value: {pval:.6f}")
    print(f"- 오즈비: {odds:.6f}")

    if pval < 0.05:
        print(f"→ {var}는 유의한 변수이다.")
    else:
        print(f"→ {var}는 유의하다고 보기 어렵다.")

    print(f"→ {var}가 1단위 증가할 때 연체 odds는 약 {odds:.6f}배가 되며, 방향은 {direction}이다.")

## 실습 2 결과 해석 가이드

샘플 CSV 기준으로는 다음과 같은 해석이 가능하다.

- `income`의 계수가 음수이면 소득이 높을수록 연체 가능성이 낮아진다.
- `debt_ratio`의 계수가 양수이면서 매우 크면 부채비율이 높을수록 연체 가능성이 크게 증가한다.
- `late_cnt`의 계수가 양수이면 과거 연체 횟수가 많을수록 다시 연체할 가능성이 높아진다.
- accuracy와 error rate를 함께 보면 분류 성능을 평가할 수 있다.
- residual deviance는 일반적으로 작을수록 적합이 더 양호하다고 본다.

### 보고서 문장 예시
“로지스틱 회귀분석 결과, `income`, `debt_ratio`, `late_cnt`는 모두 연체 여부에 유의한 영향을 미쳤다. 특히 `debt_ratio`의 오즈비가 크게 나타나 부채비율이 증가할수록 연체 발생 odds가 크게 증가한다고 해석할 수 있다. 또한 본 모형의 accuracy와 error rate를 확인한 결과, 분류 성능은 실무 적용 가능 수준으로 평가할 수 있다.”

## 학생 확인 질문 2

1. `income`의 계수가 음수라는 것은 무엇을 의미하는가?  
2. `debt_ratio`의 오즈비가 매우 크면 어떤 의미인가?  
3. `late_cnt`가 유의한 변수라면 어떤 실무적 시사점이 있는가?  
4. accuracy와 error rate는 어떤 관계를 가지는가?  
5. log-likelihood와 residual deviance는 왜 확인하는가?  

In [ ]:
# 두 실습 결과 핵심 지표 비교

summary_df = pd.DataFrame({
    'dataset': ['churn_logit1.csv', 'loan_default_logit2.csv'],
    'accuracy': [churn_acc, loan_acc],
    'error_rate': [churn_err, loan_err],
    'log_likelihood': [churn_model.llf, loan_model.llf],
    'pseudo_R2': [churn_model.prsquared, loan_model.prsquared],
    'residual_deviance': [churn_dev, loan_dev]
})

print("=== 두 모형 비교표 ===")
display(summary_df)

# 3. 최종 정리

## 이번 실습에서 배운 것
- 이진 종속변수는 로지스틱 회귀로 분석한다.
- 계수 부호는 사건 발생 방향을 의미한다.
- p-value는 변수의 유의성을 판단한다.
- 오즈비는 실무 보고서에서 가장 자주 사용하는 해석 도구이다.
- accuracy, error rate, log-likelihood, residual deviance를 함께 확인해야 한다.

## 한 줄 요약
- `churn_logit1.csv`: 고객 이탈에 영향을 주는 요인을 찾는 문제
- `loan_default_logit2.csv`: 대출 연체를 예측하고 모형 성능을 평가하는 문제

# 4. 보고서 작성 템플릿

## [실습 1 보고서 템플릿]
본 연구에서는 고객 이탈 여부를 설명하기 위해 로지스틱 회귀분석을 수행하였다.  
분석 결과, __________________ 변수는 유의한 영향을 보였고(p < 0.05),  
특히 __________________ 변수의 오즈비는 __________로 나타나  
해당 변수가 1단위 증가할 때 이탈 odds가 __________배 변한다고 해석할 수 있다.  
반면 __________________ 변수는 유의하지 않아 이탈에 직접적인 영향을 준다고 보기 어렵다.  
또한 모형의 accuracy는 __________, error rate는 __________로 나타났다.

## [실습 2 보고서 템플릿]
본 연구에서는 대출 연체 여부를 설명하기 위해 로지스틱 회귀분석을 수행하였다.  
분석 결과, __________________, __________________, __________________ 변수는  
모두 유의한 영향을 보였다.  
특히 __________________ 변수의 오즈비가 크게 나타나  
해당 변수의 증가가 연체 발생 odds를 크게 높이는 것으로 해석되었다.  
모형의 log-likelihood는 __________, residual deviance는 __________이며,  
accuracy는 __________, error rate는 __________로 나타났다.

# 5. 추가 과제

1. 예측 기준값(threshold)을 0.5가 아니라 0.4 또는 0.6으로 바꾸면 accuracy는 어떻게 달라지는가?
2. churn 데이터에서 가장 영향력이 큰 변수는 무엇인가?
3. loan default 데이터에서 실무적으로 가장 중요한 관리 변수는 무엇인가?
4. 오즈비가 1보다 작은 경우와 1보다 큰 경우를 각각 실제 문장으로 써 보라.